# Problem definition and preprocessing

Hospital-exit time alone does not identify how a stay ended. This notebook
defines the two competing events, fixes the day-3 prediction point, determines
predictor eligibility, and specifies leakage-safe preprocessing for the two
candidate model families.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import (
    CANDIDATE_CATEGORICAL_FEATURES, CANDIDATE_FEATURES,
    CANDIDATE_NUMERIC_FEATURES, FEATURE_DISPLAY_NAMES, LANDMARK,
)
from src.data import create_fixed_split, load_support2

## Competing hospital-exit events

For in-hospital deaths, follow-up ends on the hospital-exit day. After live
discharge, later follow-up continues. Hospital exit is therefore represented
by live discharge and in-hospital death; later mortality does not relabel a
live discharge.

In [2]:
df = load_support2()
train_df, _, _ = create_fixed_split(df)
train_df = train_df.set_index('patient_id')
endpoint = train_df.assign(exit_type=np.where(
    train_df['in_hospital_death'].eq(1), 'In-hospital death', 'Live discharge'
))
hospital_death = endpoint['in_hospital_death'].eq(1)
assert (endpoint.loc[hospital_death, 'follow_up_days']
        == endpoint.loc[hospital_death, 'days_to_hospital_exit']).all()
assert (endpoint.loc[~hospital_death, 'follow_up_days']
        > endpoint.loc[~hospital_death, 'days_to_hospital_exit']).all()
display(endpoint.groupby('exit_type')[[
    'days_to_hospital_exit', 'follow_up_days'
]].agg(['count', 'median', 'mean']).round(2))

days_to_hospital_exit               follow_up_days         \
                                  count median   mean          count median   
exit_type                                                                     
In-hospital death                  1661   10.0  17.05           1661   10.0   
Live discharge                     4712   11.0  18.14           4712  491.0   

                           
                     mean  
exit_type                  
In-hospital death   17.05  
Live discharge     644.67

## Day-3 landmark

Physiological, laboratory, and functional-status measurements are associated
with the third SUPPORT study day. Patients whose exit is recorded on that day
are excluded because the public data do not establish whether assessment
preceded exit. Remaining patients start follow-up on the next day.

In [3]:
train_model = train_df.loc[train_df['days_to_hospital_exit'] > LANDMARK].copy()
train_model['event'] = np.where(train_model['in_hospital_death'].eq(1), 2, 1)
display(pd.DataFrame(
    {'Patients': [
        len(train_df),
        int(train_df['days_to_hospital_exit'].eq(LANDMARK).sum()),
        len(train_model),
    ]},
    index=['Training cohort', 'Same-day exits excluded', 'At risk after day 3'],
))

,Patients
Training cohort,6373
Same-day exits excluded,233
At risk after day 3,6140


## Predictor eligibility

Eligibility asks whether information may enter development; it does not imply
predictive usefulness. Twenty identifier, outcome, follow-up, resource-use,
prognostic-score, care-process, later-outcome, or deliberately derived fields
are excluded. The remaining 28 candidates contain 20 numeric and 8 categorical
variables. Exact definitions and exclusion reasons appear in the data dictionary.

In [4]:
candidate_table = pd.DataFrame({
    'Type': ['Numeric'] * len(CANDIDATE_NUMERIC_FEATURES)
            + ['Categorical'] * len(CANDIDATE_CATEGORICAL_FEATURES),
    'Variable': [FEATURE_DISPLAY_NAMES[name] for name in CANDIDATE_FEATURES],
})
display(candidate_table)
assert len(CANDIDATE_FEATURES) == 28
assert len(CANDIDATE_NUMERIC_FEATURES) == 20
assert len(CANDIDATE_CATEGORICAL_FEATURES) == 8

,Type,Variable
0,Numeric,Age
1,Numeric,Years of education
2,Numeric,Number of comorbidities
3,Numeric,Hospital day at study entry
4,Numeric,Mean arterial pressure
5,Numeric,White blood cell count
6,Numeric,Heart rate
7,Numeric,Respiratory rate
8,Numeric,Temperature
9,Numeric,PaO2/FiO2 ratio


## Model-specific preprocessing

All transformations are fitted inside the relevant training fold. Numeric
measurements receive median imputation and explicit missingness indicators;
categorical variables receive a Missing level and safe one-hot encoding.
Standardisation is used for logistic regression because coefficient
regularisation depends on scale, but omitted for gradient boosting.


In [5]:
display(pd.DataFrame([
    {
        'Model family': 'Multinomial logistic regression',
        'Numeric processing': 'Median, indicators, standardisation',
        'Categorical processing': 'Missing level, one-hot encoding',
    },
    {
        'Model family': 'Histogram gradient boosting',
        'Numeric processing': 'Median, indicators, original scale',
        'Categorical processing': 'Missing level, one-hot encoding',
    },
]))


,Model family,Numeric processing,Categorical processing
0,Multinomial logistic regression,"Median, indicators, standardisation","Missing level, one-hot encoding"
1,Histogram gradient boosting,"Median, indicators, original scale","Missing level, one-hot encoding"


Missingness indicators are predictive inputs, not explanations for why data are
absent. Notebook 04 separately examines whether the strong ADL signal arises
from observed patient-reported values, their missingness pattern, or both.